In [ ]:
"Shor's Algorithm - Scaling Analysis for Classical Period Finding"
import math
import os
import numpy as np
import time
import random
import pandas as pd

def gcd(a, b):
    """Euclidean algorithm for greatest common divisor."""
    while b:
        a, b = b, a % b
    return a

def is_prime(n):
    """Trial division primality test."""
    if n < 2 or (n > 2 and n % 2 == 0):
        return False
    if n == 2:
        return True
    for i in range(3, int(math.sqrt(n)) + 1, 2):
        if n % i == 0:
            return False
    return True

def factorize(N):
    """Return list of prime factors (trial division)."""
    factors = []
    d, temp = 2, N
    while d * d <= temp:
        while temp % d == 0:
            factors.append(d)
            temp //= d
        d += 1
    if temp > 1:
        factors.append(temp)
    return factors

def is_semiprime(n):
    """True if n = p × q for distinct primes (RSA-style)."""
    f = factorize(n)
    return len(f) == 2 and f[0] != f[1]

def classical_order(x, N):
    """Find smallest r with x^r ≡ 1 (mod N). Returns (r, op_count, time_seconds)."""
    r, value, ops = 1, x % N, 0
    start = time.perf_counter()
    while value != 1:
        value = (value * x) % N
        r, ops = r + 1, ops + 1
        if r > N:
            elapsed = time.perf_counter() - start
            return None, ops, elapsed
    elapsed = time.perf_counter() - start
    return r, ops, elapsed

def analyze_single_iteration(N, a):
    """One Shor iteration: pick base a, find order, extract factors if possible."""
    ops = 1
    period_time = 0.0
    if gcd(a, N) != 1:  # Lucky: a shares factor with N
        return {'success': True, 'operations': ops, 'period_time': period_time}
    r, o, period_time = classical_order(a, N)
    ops += o
    if r is None or r % 2 != 0:  # Need even order
        return {'success': False, 'operations': ops, 'period_time': period_time}
    x_half = pow(a, r // 2, N)
    if x_half == N - 1:  # x^(r/2) ≡ -1: no useful factors
        return {'success': False, 'operations': ops + 1, 'period_time': period_time}
    p, q = gcd(x_half - 1, N), gcd(x_half + 1, N)  # Extract factors
    success = (1 < p < N) or (1 < q < N)
    return {'success': success, 'operations': ops + 3, 'period_time': period_time}

def run_iterations(N, num_iterations=10):
    """Run Shor iterations with random coprime bases; return stats with per-iteration data."""
    bases = [a for a in range(2, min(N, 10000)) if gcd(a, N) == 1]
    if not bases:
        return None
    
    # Warm-up run to avoid JIT/startup overhead skewing first timed iteration
    _ = analyze_single_iteration(N, random.choice(bases))
    
    success_count = 0
    ops_list = []
    period_time_list = []
    
    for _ in range(num_iterations):
        r = analyze_single_iteration(N, random.choice(bases))
        ops_list.append(r['operations'])
        period_time_list.append(r['period_time'] * 1000)  # Convert to ms
        if r['success']:
            success_count += 1
    
    ops_arr = np.array(ops_list)
    period_time_arr = np.array(period_time_list)
    
    return {
        'N': N, 'num_bits': N.bit_length(), 'num_digits': len(str(N)),
        'factors': factorize(N), 'num_iterations': num_iterations,
        'success_count': success_count, 'success_rate': 100 * success_count / num_iterations,
        'total_operations': int(ops_arr.sum()),
        'avg_operations': ops_arr.mean(),
        'std_operations': ops_arr.std(),
        'min_operations': int(ops_arr.min()),
        'max_operations': int(ops_arr.max()),
        'avg_period_time_ms': period_time_arr.mean(),
        'std_period_time_ms': period_time_arr.std(),
    }

def generate_semiprimes(min_n, max_n, num_samples):
    """Generate semiprimes evenly distributed across bit lengths."""
    min_bits = min_n.bit_length()
    max_bits = max_n.bit_length()
    
    # First pass: find all semiprimes at each bit level to know limits
    all_by_bits = {}
    for bits in range(min_bits, max_bits + 1):
        low = max(min_n, 2**(bits-1))
        high = min(max_n, 2**bits - 1)
        if high - low < 200000:
            all_by_bits[bits] = sorted([n for n in range(low, high+1) if is_semiprime(n)])
        else:
            all_by_bits[bits] = None  # Too large to enumerate; will sample
    
    # Allocate samples per level, redistributing leftover from small levels
    result = []
    remaining_quota = num_samples
    remaining_levels = max_bits - min_bits + 1
    
    for bits in range(min_bits, max_bits + 1):
        target = remaining_quota // remaining_levels
        low = max(min_n, 2**(bits-1))
        high = min(max_n, 2**bits - 1)
        
        if all_by_bits[bits] is not None:
            available = all_by_bits[bits]
            if len(available) <= target:
                chosen = available
            else:
                indices = np.linspace(0, len(available)-1, target, dtype=int)
                chosen = [available[i] for i in indices]
        else:
            targets = np.linspace(low, high, target + 10)
            found, seen = [], set()
            for t in targets:
                start = int(t) | 1
                for i in range(5000):
                    for c in [start + 2*i, max(low, start - 2*i)]:
                        if low <= c <= high and is_semiprime(c) and c not in seen:
                            found.append(c)
                            seen.add(c)
                            break
                    else:
                        continue
                    break
                if len(found) >= target:
                    break
            chosen = sorted(found)[:target]
        
        result.extend(chosen)
        remaining_quota -= len(chosen)
        remaining_levels -= 1
    
    return sorted(result)

def extrapolate(results):
    """Fit ops = a×N^b; extrapolate to RSA key sizes."""
    N_arr = np.array([r['N'] for r in results])
    log_N, log_ops = np.log10(N_arr), np.log10(np.array([r['avg_operations'] for r in results]))
    slope, intercept = np.polyfit(log_N, log_ops, 1)
    a = 10**intercept
    r2 = 1 - np.sum((log_ops - (slope*log_N + intercept))**2) / np.sum((log_ops - np.mean(log_ops))**2)
    
    ref = results[-1]
    ref_time, ref_ops = ref['avg_period_time_ms']/1000, ref['avg_operations']
    sec_per_year = 365.25 * 24 * 3600
    rsa = {}
    for name, bits in [('RSA-512', 512), ('RSA-1024', 1024), ('RSA-2048', 2048), ('RSA-4096', 4096)]:
        log_N_rsa = bits * np.log10(2)
        log_ops_rsa = intercept + slope * log_N_rsa
        log_time = log_ops_rsa + np.log10(ref_time) - np.log10(ref_ops) if ref_ops else -9
        log_years = log_time - np.log10(sec_per_year)
        rsa[name] = {'ops': log_ops_rsa, 'years': log_years}
    
    return {'exponent': slope, 'coefficient': a, 'log_a': intercept, 'r_squared': r2, 'rsa': rsa,
            'fitted_log_ops': intercept + slope * log_N, 'log_N': log_N, 'log_ops': log_ops}

def run_analysis():
    """Run the full analysis. Returns (results, fit, config) for use in export."""
    MIN_N, MAX_N = 15, 100_000_000
    BIT_SPLIT = 14  # Boundary between small and large
    SAMPLES_SMALL, ITERS_SMALL = 1000, 50   # 4-14 bits
    SAMPLES_LARGE, ITERS_LARGE = 1000, 50   # 15-27 bits
    
    # Generate semiprimes for each range separately
    small_max = 2**BIT_SPLIT - 1  # 16383
    large_min = 2**BIT_SPLIT      # 16384
    
    print(f"Generating semiprimes: {SAMPLES_SMALL} small (4-{BIT_SPLIT} bits), {SAMPLES_LARGE} large ({BIT_SPLIT+1}-27 bits)")
    N_small = generate_semiprimes(MIN_N, small_max, SAMPLES_SMALL)
    N_large = generate_semiprimes(large_min, MAX_N, SAMPLES_LARGE)
    print(f"Generated {len(N_small)} small + {len(N_large)} large = {len(N_small)+len(N_large)} total")
    
    results = []
    total = len(N_small) + len(N_large)
    start = time.time()
    
    # Run small N with fewer iterations
    for i, N in enumerate(N_small, 1):
        if i % 50 == 0 or i == 1:
            pct = 100 * i / total
            eta = (time.time() - start) / i * (total - i) if i else 0
            print(f"[{i}/{total}] {pct:.1f}% | Small range | ETA: {eta:.0f}s")
        r = run_iterations(N, ITERS_SMALL)
        if r and r['success_rate'] > 0:
            results.append(r)
    
    # Run large N with more iterations
    for i, N in enumerate(N_large, 1):
        j = len(N_small) + i
        if i % 10 == 0 or i == 1:
            pct = 100 * j / total
            eta = (time.time() - start) / j * (total - j) if j else 0
            print(f"[{j}/{total}] {pct:.1f}% | Large range | ETA: {eta:.0f}s")
        r = run_iterations(N, ITERS_LARGE)
        if r and r['success_rate'] > 0:
            results.append(r)
    
    fit = extrapolate(results)
    elapsed = time.time() - start
    print(f"\nDone in {elapsed:.1f}s")
    print(f"Scaling: O(N^{fit['exponent']:.2f}), R² = {fit['r_squared']:.3f}")
    print(f"RSA-2048: ~10^{fit['rsa']['RSA-2048']['ops']:.0f} ops, ~10^{fit['rsa']['RSA-2048']['years']:.0f} years")
    
    config = {
        'BIT_SPLIT': BIT_SPLIT,
        'N_small_count': len(N_small),
        'N_large_count': len(N_large),
        'ITERS_SMALL': ITERS_SMALL,
        'ITERS_LARGE': ITERS_LARGE,
    }
    
    return results, fit, config

def export_to_excel(results, fit, config, excel_path=None):
    """Export results to Excel. Can be run independently after run_analysis()."""
    if excel_path is None:
        cwd = os.getcwd()
        excel_path = os.path.join(cwd, "Classical_Shor_Analysis_Results.xlsx") if os.path.basename(cwd) == "Comparison analysis circuits" else os.path.join(cwd, "Comparison analysis circuits", "Classical_Shor_Analysis_Results.xlsx")
    
    BIT_SPLIT = config['BIT_SPLIT']
    
    try:
        with pd.ExcelWriter(excel_path, engine='openpyxl') as w:
            # Sheet 1: Full results
            pd.DataFrame([{
                'N': r['N'],
                'Bits': r['num_bits'],
                'Digits': r['num_digits'],
                'Factors': 'x'.join(map(str, r['factors'])),
                'Iterations': r['num_iterations'],
                'Success_Rate_%': r['success_rate'],
                'Avg_Ops': round(r['avg_operations'], 2),
                'Std_Ops': round(r['std_operations'], 2),
                'Min_Ops': r['min_operations'],
                'Max_Ops': r['max_operations'],
                'Avg_Period_Time_ms': round(r['avg_period_time_ms'], 6),
                'Std_Period_Time_ms': round(r['std_period_time_ms'], 6),
                'Log10_N': round(fit['log_N'][i], 4),
                'Log10_Avg_Ops': round(fit['log_ops'][i], 4),
                'Log10_Avg_Ops_Fitted': round(fit['fitted_log_ops'][i], 4),
                'Log10_Avg_Period_Time_s': round(np.log10(r['avg_period_time_ms'] / 1000), 4) if r['avg_period_time_ms'] > 0 else None
            } for i, r in enumerate(results)]).to_excel(w, sheet_name='Results', index=False)
            
            # Sheet 2: RSA extrapolation
            pd.DataFrame([{
                'Key': k,
                'Bits': bits,
                'Log10_N': round(bits * np.log10(2), 1),
                'Log10_Ops': round(v['ops'], 1),
                'Log10_Years': round(v['years'], 1)
            } for (k, v), bits in zip(fit['rsa'].items(), [512, 1024, 2048, 4096])]).to_excel(w, sheet_name='RSA_Extrapolation', index=False)
            
            # Sheet 3: Fit parameters
            pd.DataFrame([
                {'Parameter': 'Exponent (b)', 'Value': round(fit['exponent'], 4), 'Description': 'Slope of log-log fit; ops scales as N^b'},
                {'Parameter': 'Coefficient (a)', 'Value': round(fit['coefficient'], 6), 'Description': 'Multiplier in ops = a * N^b'},
                {'Parameter': 'Log10(a)', 'Value': round(fit['log_a'], 4), 'Description': 'Y-intercept of log-log fit'},
                {'Parameter': 'R_squared', 'Value': round(fit['r_squared'], 4), 'Description': 'Goodness of fit (1.0 = perfect)'},
                {'Parameter': 'N_min', 'Value': results[0]['N'], 'Description': 'Smallest semiprime tested'},
                {'Parameter': 'N_max', 'Value': results[-1]['N'], 'Description': 'Largest semiprime tested'},
                {'Parameter': 'Num_Samples_Total', 'Value': len(results), 'Description': 'Total semiprimes tested'},
                {'Parameter': 'Num_Samples_Small', 'Value': config['N_small_count'], 'Description': f'Semiprimes in 4-{BIT_SPLIT} bit range'},
                {'Parameter': 'Num_Samples_Large', 'Value': config['N_large_count'], 'Description': f'Semiprimes in {BIT_SPLIT+1}-27 bit range'},
                {'Parameter': 'Iterations_Small', 'Value': config['ITERS_SMALL'], 'Description': f'Iterations per semiprime (4-{BIT_SPLIT} bits)'},
                {'Parameter': 'Iterations_Large', 'Value': config['ITERS_LARGE'], 'Description': f'Iterations per semiprime ({BIT_SPLIT+1}-27 bits)'},
            ]).to_excel(w, sheet_name='Fit_Parameters', index=False)
            
        print(f"Exported: {excel_path}")
    except Exception as e:
        print(f"Excel export failed: {e}")

def main():
    results, fit, config = run_analysis()
    export_to_excel(results, fit, config)

if __name__ == "__main__":
    main()

Generating semiprimes: 1000 small (4-14 bits), 1000 large (15-27 bits)
Generated 1000 small + 1000 large = 2000 total
[1/2000] 0.1% | Small range | ETA: 0s
[50/2000] 2.5% | Small range | ETA: 0s
[100/2000] 5.0% | Small range | ETA: 0s
[150/2000] 7.5% | Small range | ETA: 0s
[200/2000] 10.0% | Small range | ETA: 0s
[250/2000] 12.5% | Small range | ETA: 0s
[300/2000] 15.0% | Small range | ETA: 1s
[350/2000] 17.5% | Small range | ETA: 1s
[400/2000] 20.0% | Small range | ETA: 1s
[450/2000] 22.5% | Small range | ETA: 1s
[500/2000] 25.0% | Small range | ETA: 1s
[550/2000] 27.5% | Small range | ETA: 1s
[600/2000] 30.0% | Small range | ETA: 1s
[650/2000] 32.5% | Small range | ETA: 1s
[700/2000] 35.0% | Small range | ETA: 1s
[750/2000] 37.5% | Small range | ETA: 1s
[800/2000] 40.0% | Small range | ETA: 2s
[850/2000] 42.5% | Small range | ETA: 2s
[900/2000] 45.0% | Small range | ETA: 2s
[950/2000] 47.5% | Small range | ETA: 2s
[1000/2000] 50.0% | Small range | ETA: 2s
[1001/2000] 50.0% | Large r